In [2]:
import os
import json
import cv2
import random
import shutil

# Define dataset paths
json_path = "/home/idrone2/Desktop/rsm.json"  # Update this
images_dir = "/home/idrone2/Desktop/rsm_annotated"  # Update this
output_dir = "/home/idrone2/Desktop/Ranjith-works/yolo/yolo_dataset_2"

# Create train/val image and label folders
train_images_dir = os.path.join(output_dir, "images/train")
val_images_dir = os.path.join(output_dir, "images/val")
train_labels_dir = os.path.join(output_dir, "labels/train")
val_labels_dir = os.path.join(output_dir, "labels/val")

os.makedirs(train_images_dir, exist_ok=True)
os.makedirs(val_images_dir, exist_ok=True)
os.makedirs(train_labels_dir, exist_ok=True)
os.makedirs(val_labels_dir, exist_ok=True)

# Load JSON file
with open(json_path, "r") as f:
    data = json.load(f)

# Get image filenames
image_filenames = list(data.keys())
random.shuffle(image_filenames)

# Split dataset (80% train, 20% val)
train_size = int(0.8 * len(image_filenames))
train_images = image_filenames[:train_size]
val_images = image_filenames[train_size:]

# Class mapping (Ensure correct order)
class_names = ["Red Spider Mite"]
class_mapping = {name: idx for idx, name in enumerate(class_names)}

# Function to normalize points
def normalize_points(points, img_width, img_height):
    return [(x / img_width, y / img_height) for x, y in points]

# Process each image annotation
for image_name in image_filenames:
    image_path = os.path.join(images_dir, image_name)
    img = cv2.imread(image_path)

    if img is None:
        continue

    img_height, img_width, _ = img.shape
    label_file_name = image_name.replace(".jpg", ".txt").replace(".JPG", ".txt")

    if image_name in train_images:
        shutil.copy(image_path, os.path.join(train_images_dir, image_name))
        label_file = os.path.join(train_labels_dir, label_file_name)
    else:
        shutil.copy(image_path, os.path.join(val_images_dir, image_name))
        label_file = os.path.join(val_labels_dir, label_file_name)

    with open(label_file, "w") as label_out:
        for region in data[image_name]["regions"].values():
            shape = region["shape_attributes"]
            if shape["name"] == "polygon":
                all_points_x = shape["all_points_x"]
                all_points_y = shape["all_points_y"]

                # Normalize polygon points
                normalized_polygon = normalize_points(
                    list(zip(all_points_x, all_points_y)), img_width, img_height
                )

                # Get class label
                class_label = region["region_attributes"]["label"]
                class_id = class_mapping[class_label]

                # Convert to YOLOv11 segmentation format: class_id x1 y1 x2 y2 ... xn yn
                polygon_str = " ".join([f"{x} {y}" for x, y in normalized_polygon])
                label_out.write(f"{class_id} {polygon_str}\n")

print("✅ Conversion to YOLOv11 segmentation format completed!")


✅ Conversion to YOLOv11 segmentation format completed!


[ WARN:0@100.385] global loadsave.cpp:268 findDecoder imread_('/home/idrone2/Desktop/rsm_annotated/annotations'): can't open/read file: check file path/integrity
[ WARN:0@100.385] global loadsave.cpp:268 findDecoder imread_('/home/idrone2/Desktop/rsm_annotated/categories'): can't open/read file: check file path/integrity
[ WARN:0@100.385] global loadsave.cpp:268 findDecoder imread_('/home/idrone2/Desktop/rsm_annotated/images'): can't open/read file: check file path/integrity
[ WARN:0@100.385] global loadsave.cpp:268 findDecoder imread_('/home/idrone2/Desktop/rsm_annotated/info'): can't open/read file: check file path/integrity


In [4]:
import os
import json
import cv2
import random
import shutil

# Define dataset paths
json_path = "/home/idrone2/Desktop/rsm.json"  # Update this
images_dir = "/home/idrone2/Desktop/rsm_annotated"  # Update this
output_dir = "/home/idrone2/Desktop/Ranjith-works/yolo/yolo_dataset_2"

# Create train/val image and label folders
train_images_dir = os.path.join(output_dir, "images/train")
val_images_dir = os.path.join(output_dir, "images/val")
train_labels_dir = os.path.join(output_dir, "labels/train")
val_labels_dir = os.path.join(output_dir, "labels/val")

os.makedirs(train_images_dir, exist_ok=True)
os.makedirs(val_images_dir, exist_ok=True)
os.makedirs(train_labels_dir, exist_ok=True)
os.makedirs(val_labels_dir, exist_ok=True)

# Load JSON file
with open(json_path, "r") as f:
    data = json.load(f)

# Extract image filenames from "images" section
images = {img["id"]: img["file_name"] for img in data["images"]}
random.shuffle(list(images.values()))

# Split dataset (80% train, 20% val)
train_size = int(0.8 * len(images))
train_images = list(images.values())[:train_size]
val_images = list(images.values())[train_size:]

# Class mapping (Ensure correct order)
class_names = ["Red Spider Mite"]
class_mapping = {name: idx for idx, name in enumerate(class_names)}

# Function to normalize points
def normalize_points(points, img_width, img_height):
    return [(x / img_width, y / img_height) for x, y in points]

# Process each image annotation
for annotation in data["annotations"]:
    image_id = annotation["image_id"]
    if image_id not in images:
        continue  # Skip if image ID not found in dataset

    image_name = images[image_id]
    image_path = os.path.join(images_dir, image_name)
    img = cv2.imread(image_path)

    if img is None:
        print(f"⚠️ Warning: Cannot open {image_path}")
        continue

    img_height, img_width = img.shape[:2]
    label_file_name = image_name.replace(".jpg", ".txt").replace(".JPG", ".txt")

    if image_name in train_images:
        shutil.copy(image_path, os.path.join(train_images_dir, image_name))
        label_file = os.path.join(train_labels_dir, label_file_name)
    else:
        shutil.copy(image_path, os.path.join(val_images_dir, image_name))
        label_file = os.path.join(val_labels_dir, label_file_name)

    with open(label_file, "w") as label_out:
        if "segmentation" in annotation:
            segmentation = annotation["segmentation"][0]  # COCO segmentation is a list of lists
            normalized_polygon = normalize_points(
                [(segmentation[i], segmentation[i+1]) for i in range(0, len(segmentation), 2)], img_width, img_height
            )

            # Get class label
            class_id = annotation["category_id"] - 1  # COCO format starts category IDs from 1

            # Convert to YOLO segmentation format: class_id x1 y1 x2 y2 ... xn yn
            polygon_str = " ".join([f"{x:.6f} {y:.6f}" for x, y in normalized_polygon])
            label_out.write(f"{class_id} {polygon_str}\n")

print("✅ Conversion to YOLOv11 segmentation format completed!")


✅ Conversion to YOLOv11 segmentation format completed!


In [4]:
import os
import json
import cv2

# Define base dataset and output paths
base_dir = "/home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone"  # Update this to your dataset root
output_dir = "/home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/"

splits = ["train", "valid", "test"]

# ✅ Your class names
class_names = ["RSM_Moderate", "RSM_Severe"]
class_mapping = {name: idx for idx, name in enumerate(class_names)}

# ⚙️ Normalize polygon points for YOLO
def normalize_points(points, img_width, img_height):
    return [(x / img_width, y / img_height) for x, y in points]

for split in splits:
    split_image_dir = os.path.join(base_dir, split)
    annotation_file = os.path.join(base_dir, split, "_annotations.coco.json")

    # 🗂️ Output folders
    out_image_dir = os.path.join(output_dir, "images", split)
    out_label_dir = os.path.join(output_dir, "labels", split)
    os.makedirs(out_image_dir, exist_ok=True)
    os.makedirs(out_label_dir, exist_ok=True)

    # 📖 Load COCO JSON
    with open(annotation_file, "r") as f:
        data = json.load(f)

    # 🗺️ Map COCO image_id to file_name
    image_id_to_name = {img["id"]: img["file_name"] for img in data["images"]}

    # 🔍 Build a lookup for actual image filenames (from disk)
    actual_image_files = {
        os.path.splitext(f)[0].replace("_jpg", ".jpg"): f
        for f in os.listdir(split_image_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    }

    # 🗺️ Map category_id → class_id
    category_id_to_class_id = {}
    for cat in data["categories"]:
        if cat["name"] in class_mapping:
            category_id_to_class_id[cat["id"]] = class_mapping[cat["name"]]

    # 🖼️ Process annotations
    for annotation in data["annotations"]:
        image_id = annotation["image_id"]
        orig_image_name = image_id_to_name.get(image_id)
        if not orig_image_name:
            continue

        # 📂 Match image filename with actual hashed one
        matched_image_name = actual_image_files.get(orig_image_name, None)
        if not matched_image_name:
            print(f"⚠️ No matching image file for '{orig_image_name}' in {split_image_dir}")
            continue

        image_path = os.path.join(split_image_dir, matched_image_name)
        img = cv2.imread(image_path)
        if img is None:
            print(f"⚠️ Cannot open image: {image_path}")
            continue

        img_height, img_width = img.shape[:2]
        label_file_name = os.path.splitext(matched_image_name)[0] + ".txt"

        # 🗃️ Copy image to output folder
        out_image_path = os.path.join(out_image_dir, matched_image_name)
        if not os.path.exists(out_image_path):
            cv2.imwrite(out_image_path, img)

        # 📝 Write YOLO label
        label_path = os.path.join(out_label_dir, label_file_name)
        with open(label_path, "a") as label_out:
            if "segmentation" in annotation and annotation["segmentation"]:
                segmentation = annotation["segmentation"][0]  # COCO: [[x1, y1, x2, y2, ...]]
                normalized_polygon = normalize_points(
                    [(segmentation[i], segmentation[i+1]) for i in range(0, len(segmentation), 2)],
                    img_width, img_height
                )

                category_id = annotation["category_id"]
                class_id = category_id_to_class_id.get(category_id)
                if class_id is None:
                    print(f"⚠️ Unknown category_id: {category_id}")
                    continue

                polygon_str = " ".join([f"{x:.6f} {y:.6f}" for x, y in normalized_polygon])
                label_out.write(f"{class_id} {polygon_str}\n")

print("✅ All splits converted to YOLO segmentation format successfully!")


⚠️ No matching image file for 'RSM_09859_jpg.rf.a80db70a55be260cdb85c702efb16e86.jpg' in /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone/train
⚠️ No matching image file for 'RSM_09677_jpg.rf.fc6d313d1b8e183d185a76cdb355ba5f.jpg' in /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone/train
⚠️ No matching image file for 'RSM_12527_jpg.rf.22e5c4ebddb83e5f98fdd3e06a717823.jpg' in /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone/train
⚠️ No matching image file for 'RSM_07181_jpg.rf.ae65141117d1b65d74b8b872f1ad4311.jpg' in /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone/train
⚠️ No matching image file for 'RSM_05518_jpg.rf.a7abacf5283443218eb56072c4b9e7f4.jpg' in /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone/train
⚠️ No matching image file for 'RSM_07607_jpg.rf.4576d4a6c6b41933019016b97bbc0a08.jpg' in /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Ca

In [ ]:
from ultralytics import YOLO

# Load YOLOv11 model
model = YOLO("/home/idrone2/Desktop/Ranjith-works/yolo/yolo11s-seg.pt")  # Ensure you have the pre-trained weights

# Train the model
model.train(
    data="/home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone_v2i_yolov11/data.yaml",
    epochs=150,
    imgsz=1444,
    batch=16,
    device="cuda"  # Use "cpu" if no GPU is available
)


New https://pypi.org/project/ultralytics/8.3.167 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.65 🚀 Python-3.10.12 torch-2.5.1+cu124 CUDA:0 (NVIDIA RTX A2000 12GB, 11926MiB)
engine/trainer: task=segment, mode=train, model=/home/idrone2/Desktop/Ranjith-works/yolo/yolo11s-seg.pt, data=/home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone_v2i_yolov11/data.yaml, epochs=50, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=cuda, workers=8, project=None, name=train5, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffe

train: Scanning /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone_v2i_yolov11/train/labels... 6998 images, 2 backgrounds, 0 corrupt: 100%|██████████| 6998/6998 [00:01<00:00, 3628.32it/s]


train: New cache created: /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone_v2i_yolov11/train/labels.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


/home/idrone2/.local/lib/python3.10/site-packages/albumentations/__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.3'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
/home/idrone2/.local/lib/python3.10/site-packages/ultralytics/data/augment.py:1853: UserWarning: Argument(s) 'quality_lower' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=75, p=0.0),
val: Scanning /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone_v2i_yolov11/valid/labels... 1999 images, 1 backgrounds, 0 corrupt: 100%|██████████| 1999/1999 [00:01<00:00, 1894.37it/s]

val: New cache created: /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone_v2i_yolov11/valid/labels.cache


...

                   all       1999       2105      0.638      0.626      0.669      0.594      0.637      0.628      0.669      0.583



50 epochs completed in 2.115 hours.
Optimizer stripped from runs/segment/train5/weights/last.pt, 20.5MB
Optimizer stripped from runs/segment/train5/weights/best.pt, 20.5MB

Validating runs/segment/train5/weights/best.pt...
Ultralytics 8.3.65 🚀 Python-3.10.12 torch-2.5.1+cu124 CUDA:0 (NVIDIA RTX A2000 12GB, 11926MiB)
YOLO11s-seg summary (fused): 265 layers, 10,067,977 parameters, 0 gradients, 35.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 63/63 [00:14<00:00,  4.29it/s]


                   all       1999       2105      0.627      0.648      0.674        0.6      0.624      0.653      0.675      0.589
          RSM_Moderate        999       1023      0.673      0.664      0.711      0.637      0.667      0.666      0.709      0.628
            RSM_Severe        999       1082      0.581      0.633      0.638      0.563      0.582      0.641       0.64      0.551
Speed: 0.2ms preprocess, 4.4ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to runs/segment/train5


ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ea899bebfd0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.04104

In [1]:
from ultralytics import YOLO

# Load YOLOv11 model
model = YOLO("/home/idrone2/Desktop/Ranjith-works/yolo/yolo11s-seg.pt")  # Ensure you have the pre-trained weights

# Train the model
model.train(
    data="/home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone_v2i_yolov11/data.yaml",
    epochs=100,
    imgsz=1244,
    batch=8,
    device="cuda"  # Use "cpu" if no GPU is available
)

New https://pypi.org/project/ultralytics/8.3.168 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.65 🚀 Python-3.10.12 torch-2.5.1+cu124 CUDA:0 (NVIDIA RTX A2000 12GB, 11926MiB)
engine/trainer: task=segment, mode=train, model=/home/idrone2/Desktop/Ranjith-works/yolo/yolo11s-seg.pt, data=/home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone_v2i_yolov11/data.yaml, epochs=100, time=None, patience=100, batch=8, imgsz=1244, save=True, save_period=-1, cache=False, device=cuda, workers=8, project=None, name=train5, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buff

train: Scanning /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone_v2i_yolov11/train/labels.cache... 6998 images, 2 backgrounds, 0 corrupt: 100%|██████████| 6998/6998 [00:00<?, ?it/s]


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


/home/idrone2/.local/lib/python3.10/site-packages/albumentations/__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.3'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
/home/idrone2/.local/lib/python3.10/site-packages/ultralytics/data/augment.py:1853: UserWarning: Argument(s) 'quality_lower' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=75, p=0.0),
val: Scanning /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone_v2i_yolov11/valid/labels.cache... 1999 images, 1 backgrounds, 0 corrupt: 100%|██████████| 1999/1999 [00:00<?, ?it/s]


Plotting labels to runs/segment/train5/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: SGD(lr=0.01, momentum=0.9) with parameter groups 90 weight(decay=0.0), 101 weight(decay=0.0005), 100 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 1248 train, 1248 val
Using 8 dataloader workers
Logging results to runs/segment/train5
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss  Instances       Size


...

                   all       1999       2105      0.627      0.609        0.6       0.54      0.628       0.61      0.601      0.532



100 epochs completed in 16.333 hours.
Optimizer stripped from runs/segment/train5/weights/last.pt, 20.6MB
Optimizer stripped from runs/segment/train5/weights/best.pt, 20.6MB

Validating runs/segment/train5/weights/best.pt...
Ultralytics 8.3.65 🚀 Python-3.10.12 torch-2.5.1+cu124 CUDA:0 (NVIDIA RTX A2000 12GB, 11926MiB)
YOLO11s-seg summary (fused): 265 layers, 10,067,977 parameters, 0 gradients, 35.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 125/125 [00:38<00:00,  3.24it/s]


                   all       1999       2105      0.627      0.667      0.685      0.608      0.628      0.667      0.684      0.601
          RSM_Moderate        999       1023      0.671      0.674      0.717       0.64      0.671      0.674      0.716      0.634
            RSM_Severe        999       1082      0.584      0.659      0.653      0.575      0.584       0.66      0.652      0.567
Speed: 0.4ms preprocess, 16.7ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to runs/segment/train5


ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f1e280c1e70>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.04104

In [1]:
from ultralytics import YOLO

# Load YOLOv11 model
model = YOLO("/home/idrone2/Desktop/Ranjith-works/yolo/yolo11s-seg.pt")  # Ensure you have the pre-trained weights

# Train the model
model.train(
    data="/home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone_v2i_yolov11/data.yaml",
    epochs=50,
    imgsz=1984,
    batch=2,
    patience = 20, 
    #accumulation = 8, 
    device="cuda"  # Use "cpu" if no GPU is available
)

Ultralytics 8.3.65 🚀 Python-3.10.12 torch-2.5.1+cu124 CUDA:0 (NVIDIA RTX A2000 12GB, 11926MiB)
engine/trainer: task=segment, mode=train, model=/home/idrone2/Desktop/Ranjith-works/yolo/yolo11s-seg.pt, data=/home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone_v2i_yolov11/data.yaml, epochs=50, time=None, patience=20, batch=2, imgsz=1984, save=True, save_period=-1, cache=False, device=cuda, workers=8, project=None, name=train5, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=No

train: Scanning /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone_v2i_yolov11/train/labels.cache... 6998 images, 2 backgrounds, 0 corrupt: 100%|██████████| 6998/6998 [00:00<?, ?it/s]


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


/home/idrone2/.local/lib/python3.10/site-packages/albumentations/check_version.py:107: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
/home/idrone2/.local/lib/python3.10/site-packages/ultralytics/data/augment.py:1853: UserWarning: Argument(s) 'quality_lower' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=75, p=0.0),
val: Scanning /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone_v2i_yolov11/valid/labels.cache... 1999 images, 1 backgrounds, 0 corrupt: 100%|██████████| 1999/1999 [00:00<?, ?it/s]


Plotting labels to runs/segment/train5/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.001429, momentum=0.9) with parameter groups 90 weight(decay=0.0), 101 weight(decay=0.0005), 100 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 1984 train, 1984 val
Using 8 dataloader workers
Logging results to runs/segment/train5
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss  Instances       Size


...

                   all       1999       2105      0.638      0.639      0.666      0.534      0.637      0.641      0.667      0.556



50 epochs completed in 22.952 hours.
Optimizer stripped from runs/segment/train5/weights/last.pt, 20.8MB
Optimizer stripped from runs/segment/train5/weights/best.pt, 20.8MB

Validating runs/segment/train5/weights/best.pt...
Ultralytics 8.3.65 🚀 Python-3.10.12 torch-2.5.1+cu124 CUDA:0 (NVIDIA RTX A2000 12GB, 11926MiB)
YOLO11s-seg summary (fused): 265 layers, 10,067,977 parameters, 0 gradients, 35.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 500/500 [01:30<00:00,  5.53it/s]


                   all       1999       2105      0.624      0.641      0.665      0.534      0.625      0.643      0.666      0.559
          RSM_Moderate        999       1023      0.628      0.673      0.696      0.554      0.628      0.674      0.696      0.588
            RSM_Severe        999       1082      0.621      0.609      0.634      0.515      0.622      0.612      0.637       0.53
Speed: 1.0ms preprocess, 41.6ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to runs/segment/train5


ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e0db73617b0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.04104

In [2]:
from ultralytics import YOLO

# Load YOLOv11 model
model = YOLO("/home/idrone2/Desktop/Ranjith-works/yolo/yolo11n.pt")  # Ensure you have the pre-trained weights

# Train the model
model.train(
    data="/home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone_v2i_yolov11/data.yaml",
    epochs=50,
    imgsz=1024,
    batch=8,
    patience = 20, 
    #accumulation = 8, 
    device="cuda"  # Use "cpu" if no GPU is available
)

Ultralytics 8.3.65 🚀 Python-3.10.12 torch-2.5.1+cu124 CUDA:0 (NVIDIA RTX A2000 12GB, 11926MiB)
engine/trainer: task=detect, mode=train, model=/home/idrone2/Desktop/Ranjith-works/yolo/yolo11n.pt, data=/home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone_v2i_yolov11/data.yaml, epochs=50, time=None, patience=20, batch=8, imgsz=1024, save=True, save_period=-1, cache=False, device=cuda, workers=8, project=None, name=train4, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, s

train: Scanning /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone_v2i_yolov11/train/labels.cache... 6998 images, 2 backgrounds, 0 corrupt: 100%|██████████| 6998/6998 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))



/home/idrone2/.local/lib/python3.10/site-packages/ultralytics/data/augment.py:1853: UserWarning: Argument(s) 'quality_lower' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=75, p=0.0),
val: Scanning /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone_v2i_yolov11/valid/labels.cache... 1999 images, 1 backgrounds, 0 corrupt: 100%|██████████| 1999/1999 [00:00<?, ?it/s]


Plotting labels to runs/detect/train4/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.001429, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 1024 train, 1024 val
Using 8 dataloader workers
Logging results to runs/detect/train4
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

                   all       1999       2105      0.663      0.633      0.679      0.598



50 epochs completed in 2.414 hours.
Optimizer stripped from runs/detect/train4/weights/last.pt, 5.5MB
Optimizer stripped from runs/detect/train4/weights/best.pt, 5.5MB

Validating runs/detect/train4/weights/best.pt...
Ultralytics 8.3.65 🚀 Python-3.10.12 torch-2.5.1+cu124 CUDA:0 (NVIDIA RTX A2000 12GB, 11926MiB)
YOLO11n summary (fused): 238 layers, 2,582,737 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 125/125 [00:14<00:00,  8.35it/s]


                   all       1999       2105      0.613      0.667      0.686      0.602
          RSM_Moderate        999       1023      0.611      0.706      0.716      0.635
            RSM_Severe        999       1082      0.614      0.628      0.656      0.568
Speed: 0.3ms preprocess, 4.6ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to runs/detect/train4


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e0da9b7eb30>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [1]:
from ultralytics import YOLO

# Load YOLOv11 medium model for better capacity (switch from 's' to 'm')
model = YOLO("/home/idrone2/Desktop/Ranjith-works/yolo/yolo11m.pt")  # you can use "yolo11l.pt" if GPU allows

# Train the model
model.train(
    data="/home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone_v2i_yolov11/data.yaml",
    epochs=200,           
    imgsz=1280,           
    batch=4, 
    multi_scale=False,  # Enable multi-scale training                    
)


New https://pypi.org/project/ultralytics/8.3.169 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.65 🚀 Python-3.10.12 torch-2.5.1+cu124 CUDA:0 (NVIDIA RTX A2000 12GB, 11926MiB)
engine/trainer: task=detect, mode=train, model=/home/idrone2/Desktop/Ranjith-works/yolo/yolo11m.pt, data=/home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone_v2i_yolov11/data.yaml, epochs=200, time=None, patience=100, batch=4, imgsz=1280, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train5, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=Fa

train: Scanning /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone_v2i_yolov11/train/labels.cache... 6998 images, 2 backgrounds, 0 corrupt: 100%|██████████| 6998/6998 [00:00<?, ?it/s]


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


/home/idrone2/.local/lib/python3.10/site-packages/albumentations/__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.3'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
/home/idrone2/.local/lib/python3.10/site-packages/ultralytics/data/augment.py:1853: UserWarning: Argument(s) 'quality_lower' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=75, p=0.0),
val: Scanning /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone_v2i_yolov11/valid/labels.cache... 1999 images, 1 backgrounds, 0 corrupt: 100%|██████████| 1999/1999 [00:00<?, ?it/s]


Plotting labels to runs/detect/train5/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: SGD(lr=0.01, momentum=0.9) with parameter groups 106 weight(decay=0.0), 113 weight(decay=0.0005), 112 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 1280 train, 1280 val
Using 8 dataloader workers
Logging results to runs/detect/train5
Starting training for 200 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

     66/200      8.75G     0.6004      1.031      1.036          5       1280: 100%|██████████| 1750/1750 [15:49<00:00,  1.84it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 250/250 [01:10<00:00,  3.53it/s]

                   all       1999       2105      0.653      0.658      0.692      0.609



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     67/200      8.73G     0.5937      1.002      1.027         12       1280:  86%|████████▌ | 1503/1750 [13:33<02:13,  1.85it/s]


KeyboardInterrupt: 

In [1]:

from ultralytics import RTDETR

# Load a COCO-pretrained RT-DETR-l model
model = RTDETR("/home/idrone2/Desktop/Ranjith-works/yolo/rtdetr-l.pt")

# Display model information (optional)
model.info()

# Train the model on the COCO8 example dataset for 100 epochs
results = model.train(data="/home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone_v2i_yolov11/data.yaml", epochs=100, imgsz=1240, batch = 4, device="cuda")

# Run inference with the RT-DETR-l model on the 'bus.jpg' image
results = model("/home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone_v2i_yolov11/test/images/RSM_05046_jpg.rf.177870c9ce717c8c535ee764375ef723.jpg")

rt-detr-l summary: 449 layers, 32,970,476 parameters, 0 gradients, 108.3 GFLOPs
Ultralytics 8.3.170 🚀 Python-3.10.12 torch-2.5.1+cu124 CUDA:0 (NVIDIA RTX A2000 12GB, 11926MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone_v2i_yolov11/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1240, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/home/idrone2/Desktop/Ranjith-works/yolo/rtdetr-l.pt, mome

train: Scanning /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone_v2i_yolov11/train/labels.cache... 6998 images, 2 backgrounds, 0 corrupt: 100%|██████████| 6998/6998 [00:00<?, ?it/s]


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3073.8±2124.9 MB/s, size: 1485.5 KB)


val: Scanning /home/idrone2/Desktop/Ranjith-works/yolo/Camellia_Idrone-2/Camellia_Idrone_v2i_yolov11/valid/labels.cache... 1999 images, 1 backgrounds, 0 corrupt: 100%|██████████| 1999/1999 [00:00<?, ?it/s]


Plotting labels to runs/detect/train9/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: SGD(lr=0.01, momentum=0.9) with parameter groups 143 weight(decay=0.0), 206 weight(decay=0.0005), 226 bias(decay=0.0)
Image sizes 1248 train, 1248 val
Using 8 dataloader workers
Logging results to runs/detect/train9
Starting training for 100 epochs...

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


  0%|          | 0/1750 [00:00<?, ?it/s]../aten/src/ATen/native/cuda/IndexKernel.cu:93: operator(): block: [46,0,0], thread: [64,0,0] Assertion `-sizes[i] <= index && index < sizes[i] && "index out of bounds"` failed.
../aten/src/ATen/native/cuda/IndexKernel.cu:93: operator(): block: [46,0,0], thread: [65,0,0] Assertion `-sizes[i] <= index && index < sizes[i] && "index out of bounds"` failed.
../aten/src/ATen/native/cuda/IndexKernel.cu:93: operator(): block: [46,0,0], thread: [66,0,0] Assertion `-sizes[i] <= index && index < sizes[i] && "index out of bounds"` failed.
../aten/src/ATen/native/cuda/IndexKernel.cu:93: operator(): block: [46,0,0], thread: [67,0,0] Assertion `-sizes[i] <= index && index < sizes[i] && "index out of bounds"` failed.
../aten/src/ATen/native/cuda/IndexKernel.cu:93: operator(): block: [46,0,0], thread: [68,0,0] Assertion `-sizes[i] <= index && index < sizes[i] && "index out of bounds"` failed.
../aten/src/ATen/native/cuda/IndexKernel.cu:93: operator(): block: [46

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
